In [ ]:
from __future__ import annotations

import sys
import warnings
from dataclasses import dataclass
from pathlib import Path
from typing import Iterable, Optional

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.special import softmax
from scipy.spatial.distance import cdist
from scipy.stats import mannwhitneyu, spearmanr

from sklearn.decomposition import PCA
from sklearn.metrics import (
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    log_loss,
    roc_auc_score,
)
from sklearn.model_selection import RepeatedStratifiedKFold
from sklearn.preprocessing import StandardScaler
from statsmodels.stats.multitest import multipletests

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print("Python:", sys.version.split()[0])
print("pandas:", pd.__version__)
print("numpy:", np.__version__)

In [ ]:
DATA_DIR = Path("data")
OUTPUT_DIR = DATA_DIR / "celligner_mss_tmbh_screen"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

DEPMAP_EXPRESSION_PATH = DATA_DIR / "OmicsExpressionTPMLogp1HumanProteinCodingGenes.csv"
DEPMAP_MODEL_PATH = DATA_DIR / "Model.csv"
PRISM_RESPONSE_PATH = DATA_DIR / "secondary-screen-dose-response-curve-parameters.csv"
PRISM_TREATMENT_INFO_PATH = DATA_DIR / "secondary-screen-replicate-collapsed-treatment-info.csv"



TUMOR_DATASETS = {
    "CRC": {
        "expression_path": DATA_DIR / "CRC_TCGA_LOG2TPM.csv",
        "metadata_path": DATA_DIR / "clinical_data.csv",
        "sample_id_col": None,
        "group_col": "MSI_TMB",
        "already_log2_tpm": True,
    },
    "UCEC": {
        "expression_path": DATA_DIR / "UCEC_TCGA_LOG2TPM.csv",
        "metadata_path": DATA_DIR / "ucec_clinical.csv",
        "sample_id_col": None,
        "group_col": "MSI_TMB",
        "already_log2_tpm": True,
    },
}


EXTRA_ALIGNMENT_DATASETS = [
    
    
]

CELLIGNER_MODE = "run"  
CELLIGNER_ALIGNED_PATH = OUTPUT_DIR / "celligner_aligned_expression.pkl"
CELLIGNER_MODEL_PATH = OUTPUT_DIR / "celligner_model.pkl"
CELLIGNER_LOW_MEMORY = True

CLASS_ORDER = ["MSS/TMB-L", "MSS/TMB-H", "MSI-H"]
K_CANDIDATES = [1, 3, 5, 7]
TEMPERATURE_CANDIDATES = [0.03, 0.05, 0.10, 0.20, 0.50]
PCA_VARIANCE = 0.90
MAX_VARIABLE_FEATURES = 5000
CV_REPEATS = 10

PRIMARY_PROBABILITY_THRESHOLD = 0.60
PRIMARY_MARGIN_THRESHOLD = 0.10
THRESHOLDS_TO_TEST = [0.50, 0.60, 0.70]

MIN_MODELS_PER_GROUP = 3
MIN_MODELS_CONTINUOUS = 10
FDR_CUTOFF = 0.05

CRC_ONCOTREE_CODES = ["COAD", "READ", "MACR", "COADREAD"]
UTERINE_CARCINOMA_CODES = ["UCEC", "UCS", "UCCC", "UASC"]
UTERINE_BROAD_CODES = ["UCEC", "UCS", "UCCC", "UASC", "USARC", "ESS", "ULMS"]

In [ ]:
def read_table(path: Path, index_col: Optional[int] = 0) -> pd.DataFrame:
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"File not found: {path}")
    suffix = "".join(path.suffixes).lower()
    if suffix.endswith(".parquet"):
        return pd.read_parquet(path)
    if suffix.endswith((".pkl", ".pickle")):
        return pd.read_pickle(path)
    if suffix.endswith((".tsv", ".txt")):
        return pd.read_csv(path, sep="\t", index_col=index_col)
    return pd.read_csv(path, index_col=index_col)


def canonical_sample_id(value: object) -> str:
    value = str(value).strip().replace(".", "-")
    if value.upper().startswith("TCGA-") and len(value) >= 16:
        value = value[:16]
    return value


def clean_gene_id(value: object) -> str:
    value = str(value).strip()
    if " (" in value:
        value = value.split(" (", 1)[0]
    if "|" in value:
        value = value.split("|", 1)[0]
    if value.upper().startswith("ENSG") and "." in value:
        value = value.split(".", 1)[0]
    return value.upper()


def collapse_duplicate_genes(x: pd.DataFrame) -> pd.DataFrame:
    x = x.copy()
    x.columns = [clean_gene_id(c) for c in x.columns]
    return x.T.groupby(level=0).mean().T


def normalize_group_label(value: object):
    if pd.isna(value):
        return np.nan
    key = str(value).strip().upper().replace("_", "-").replace(" ", "")
    mapping = {
        "MSS/TMB-L": "MSS/TMB-L", "MSS-TMB-L": "MSS/TMB-L", "MSS/TMBL": "MSS/TMB-L",
        "MSS/TMB-H": "MSS/TMB-H", "MSS-TMB-H": "MSS/TMB-H", "MSS/TMBH": "MSS/TMB-H",
        "MSI-H": "MSI-H", "MSIH": "MSI-H", "MSI/H": "MSI-H",
    }
    return mapping.get(key, str(value).strip())


def load_metadata(path, cancer, group_col, sample_id_col=None):
    meta = read_table(path, index_col=0 if sample_id_col is None else None)
    ids = pd.Series(meta.index, index=meta.index) if sample_id_col is None else meta[sample_id_col]
    if group_col not in meta.columns:
        raise KeyError(f"{group_col!r} not found in {path}. Columns: {list(meta.columns)}")
    out = pd.DataFrame({
        "sample_id": ids.map(canonical_sample_id).values,
        "cancer": cancer,
        "group": meta[group_col].map(normalize_group_label).values,
    })
    out = out[out["group"].isin(CLASS_ORDER)].drop_duplicates("sample_id")
    out["aligned_id"] = "TCGA::" + out["sample_id"]
    return out.set_index("sample_id", drop=False)


def orient_to_samples(x: pd.DataFrame, expected_ids=None):
    x = x.copy()
    x.index = x.index.map(canonical_sample_id)
    x.columns = [canonical_sample_id(c) for c in x.columns]
    if expected_ids is None:
        return x
    expected = {canonical_sample_id(i) for i in expected_ids}
    row_overlap = len(expected.intersection(x.index))
    col_overlap = len(expected.intersection(x.columns))
    if col_overlap > row_overlap:
        x = x.T
        x.index = x.index.map(canonical_sample_id)
    if max(row_overlap, col_overlap) == 0:
        warnings.warn("No metadata IDs matched either expression axis; assuming samples are rows.")
    return x


def load_tumor_expression(path, metadata_ids, already_log2_tpm):
    x = orient_to_samples(read_table(path, index_col=0), metadata_ids)
    x = x.apply(pd.to_numeric, errors="coerce")
    if not already_log2_tpm:
        x = np.log2(x.clip(lower=0) + 1)
    x = collapse_duplicate_genes(x)
    return x.groupby(level=0).mean()


def load_depmap_expression(path):
    x = read_table(path, index_col=0)
    if "MODELID" in x.columns:
        x = x.set_index("MODELID")
    elif "ModelID" in x.columns:
        x = x.set_index("ModelID")
    x.index = x.index.astype(str)
    x = x.drop(columns=["MODELID", "ModelID"], errors="ignore")
    x = x.apply(pd.to_numeric, errors="coerce")
    return collapse_duplicate_genes(x).groupby(level=0).mean()

In [ ]:
depmap_expression = load_depmap_expression(DEPMAP_EXPRESSION_PATH)
models = read_table(DEPMAP_MODEL_PATH, index_col=None)
prism = read_table(PRISM_RESPONSE_PATH, index_col=None)
treatment_info = read_table(PRISM_TREATMENT_INFO_PATH, index_col=None)

required_model_columns = {"ModelID", "OncotreeCode"}
missing = required_model_columns.difference(models.columns)
if missing:
    raise KeyError(f"Missing Model.csv columns: {sorted(missing)}")

tumor_expression_parts = []
tumor_metadata_parts = []

for cancer, config in TUMOR_DATASETS.items():
    meta = load_metadata(
        config["metadata_path"], cancer, config["group_col"], config["sample_id_col"]
    )
    expr = load_tumor_expression(
        config["expression_path"], meta["sample_id"], config["already_log2_tpm"]
    )
    common_ids = meta.index.intersection(expr.index)
    if len(common_ids) == 0:
        raise ValueError(f"No matched metadata/expression samples for {cancer}.")
    meta = meta.loc[common_ids].copy()
    expr = expr.loc[common_ids].copy()
    expr.index = "TCGA::" + expr.index
    tumor_metadata_parts.append(meta)
    tumor_expression_parts.append(expr)

tumor_metadata = pd.concat(tumor_metadata_parts)
labeled_tumor_expression = pd.concat(tumor_expression_parts, axis=0, join="outer")

extra_parts = []
for config in EXTRA_ALIGNMENT_DATASETS:
    extra = load_tumor_expression(
        config["expression_path"], None, config["already_log2_tpm"]
    )
    extra.index = "EXTRA::" + str(config["name"]) + "::" + extra.index.astype(str)
    extra_parts.append(extra)

all_tumor_expression = pd.concat(
    [labeled_tumor_expression] + extra_parts, axis=0, join="outer"
)

print("DepMap:", depmap_expression.shape)
print("Labeled tumors:", labeled_tumor_expression.shape)
print("All alignment tumors:", all_tumor_expression.shape)
display(tumor_metadata.groupby(["cancer", "group"]).size().rename("n").reset_index())

In [ ]:
common_genes = depmap_expression.columns.intersection(all_tumor_expression.columns)
if len(common_genes) < 1000:
    raise ValueError(
        f"Only {len(common_genes)} overlapping genes. Harmonize gene identifiers first."
    )

ccle_input = depmap_expression.loc[:, common_genes].copy()
ccle_input.index = "CL::" + ccle_input.index.astype(str)
tumor_input = all_tumor_expression.loc[:, common_genes].copy()

ccle_input = ccle_input.fillna(ccle_input.median())
tumor_input = tumor_input.fillna(tumor_input.median())
usable = (
    ccle_input.notna().all() & tumor_input.notna().all()
    & ((ccle_input.var() > 0) | (tumor_input.var() > 0))
)
ccle_input = ccle_input.loc[:, usable]
tumor_input = tumor_input.loc[:, usable]

expected_ids = set(ccle_input.index).union(tumor_input.index)

def normalize_aligned_output(x):
    x = pd.DataFrame(x).copy()
    row_overlap = len(expected_ids.intersection(x.index.astype(str)))
    col_overlap = len(expected_ids.intersection(x.columns.astype(str)))
    if col_overlap > row_overlap:
        x = x.T
    x.index = x.index.astype(str)
    return x.apply(pd.to_numeric, errors="coerce")

if CELLIGNER_MODE == "run":
    try:
        from celligner import Celligner
    except ImportError as exc:
        raise ImportError(
            "Celligner is not installed. Install it or set CELLIGNER_MODE='load'."
        ) from exc
    try:
        aligner = Celligner(low_memory=CELLIGNER_LOW_MEMORY)
    except TypeError:
        aligner = Celligner()
        if hasattr(aligner, "low_memory"):
            aligner.low_memory = CELLIGNER_LOW_MEMORY
    aligner.fit(ccle_input)
    aligner.transform(tumor_input)
    aligned_expression = normalize_aligned_output(aligner.combined_output)
    aligned_expression.to_pickle(CELLIGNER_ALIGNED_PATH)
    try:
        aligner.save(str(CELLIGNER_MODEL_PATH))
    except Exception as exc:
        warnings.warn(f"Could not save Celligner model: {exc}")
elif CELLIGNER_MODE == "load":
    aligned_expression = normalize_aligned_output(pd.read_pickle(CELLIGNER_ALIGNED_PATH))
else:
    raise ValueError("CELLIGNER_MODE must be 'run' or 'load'.")

print("Aligned matrix:", aligned_expression.shape)
print("Expected samples missing:", len(expected_ids.difference(aligned_expression.index)))

In [ ]:
def plot_source_pca(matrix, max_features=3000):
    source = pd.Series(index=matrix.index, dtype=object)
    source.loc[source.index.str.startswith("CL::")] = "Cell line"
    source.loc[source.index.str.startswith("TCGA::")] = "TCGA tumor"
    source.loc[source.index.str.startswith("EXTRA::")] = "Other tumor"

    ids = source.dropna().index.intersection(matrix.index)
    x = matrix.loc[ids].copy()
    features = x.var().sort_values(ascending=False).head(max_features).index
    x = x[features].fillna(x[features].median())
    coordinates = PCA(n_components=2, random_state=RANDOM_STATE).fit_transform(
        StandardScaler().fit_transform(x)
    )

    plt.figure(figsize=(8, 6))
    for label in source.loc[ids].unique():
        mask = source.loc[ids].values == label
        plt.scatter(coordinates[mask, 0], coordinates[mask, 1], s=18, alpha=0.65, label=label)
    plt.xlabel("PC1")
    plt.ylabel("PC2")
    plt.title("Celligner-aligned expression")
    plt.legend(frameon=False)
    plt.tight_layout()
    plt.show()

plot_source_pca(aligned_expression)

In [ ]:
@dataclass
class BalancedNearestGroupClassifier:
    k: int = 3
    temperature: float = 0.10
    pca_variance: float = 0.90
    max_features: int = 5000
    class_order: tuple = tuple(CLASS_ORDER)

    def fit(self, x, y):
        x = x.copy().apply(pd.to_numeric, errors="coerce")
        y = pd.Series(y, index=x.index).astype(str)
        missing = set(self.class_order).difference(set(y))
        if missing:
            raise ValueError(f"Training data lack classes: {sorted(missing)}")

        variances = x.var().sort_values(ascending=False)
        self.features_ = variances[np.isfinite(variances) & (variances > 0)].head(
            self.max_features
        ).index.tolist()
        self.medians_ = x[self.features_].median()
        imputed = x[self.features_].fillna(self.medians_)
        self.scaler_ = StandardScaler()
        scaled = self.scaler_.fit_transform(imputed)
        self.pca_ = PCA(n_components=self.pca_variance, svd_solver="full")
        self.training_coordinates_ = self.pca_.fit_transform(scaled)
        self.training_labels_ = y.to_numpy()
        return self

    def _transform(self, x):
        x = x.copy().apply(pd.to_numeric, errors="coerce")
        for feature in set(self.features_).difference(x.columns):
            x[feature] = np.nan
        scaled = self.scaler_.transform(x[self.features_].fillna(self.medians_))
        return self.pca_.transform(scaled)

    def class_distances(self, x):
        distances = cdist(self._transform(x), self.training_coordinates_, metric="cosine")
        outputs = []
        for class_name in self.class_order:
            class_values = distances[:, self.training_labels_ == class_name]
            k_used = min(self.k, class_values.shape[1])
            nearest = np.partition(class_values, kth=k_used - 1, axis=1)[:, :k_used]
            outputs.append(nearest.mean(axis=1))
        return pd.DataFrame(np.column_stack(outputs), index=x.index, columns=self.class_order)

    def predict_proba(self, x):
        distances = self.class_distances(x)
        probabilities = softmax(-distances.to_numpy() / self.temperature, axis=1)
        return pd.DataFrame(probabilities, index=x.index, columns=self.class_order)


def make_cv(y):
    n_splits = min(5, int(y.value_counts().min()))
    if n_splits < 2:
        raise ValueError("At least two tumors per class are required.")
    return RepeatedStratifiedKFold(
        n_splits=n_splits, n_repeats=CV_REPEATS, random_state=RANDOM_STATE
    )


def tune_classifier(x, y):
    rows = []
    cv = make_cv(y)
    for k in K_CANDIDATES:
        for temperature in TEMPERATURE_CANDIDATES:
            ba, mf1, losses = [], [], []
            for train, test in cv.split(x, y):
                model = BalancedNearestGroupClassifier(
                    k=k, temperature=temperature,
                    pca_variance=PCA_VARIANCE,
                    max_features=MAX_VARIABLE_FEATURES,
                ).fit(x.iloc[train], y.iloc[train])
                probs = model.predict_proba(x.iloc[test])
                pred = probs.idxmax(axis=1)
                ba.append(balanced_accuracy_score(y.iloc[test], pred))
                mf1.append(f1_score(y.iloc[test], pred, labels=CLASS_ORDER,
                                    average="macro", zero_division=0))
                metric_labels = sorted(CLASS_ORDER)
                losses.append(
                    log_loss(
                        y.iloc[test],
                        probs[metric_labels],
                        labels=metric_labels,
                    )
                )
            rows.append({
                "k": k, "temperature": temperature,
                "balanced_accuracy": np.mean(ba),
                "macro_f1": np.mean(mf1),
                "log_loss": np.mean(losses),
            })
    return pd.DataFrame(rows).sort_values(
        ["balanced_accuracy", "macro_f1", "log_loss"],
        ascending=[False, False, True],
    ).reset_index(drop=True)


def repeated_oof(x, y, k, temperature):
    cv = make_cv(y)
    total = pd.DataFrame(0.0, index=x.index, columns=CLASS_ORDER)
    counts = pd.Series(0, index=x.index, dtype=int)
    for train, test in cv.split(x, y):
        model = BalancedNearestGroupClassifier(
            k=k, temperature=temperature,
            pca_variance=PCA_VARIANCE,
            max_features=MAX_VARIABLE_FEATURES,
        ).fit(x.iloc[train], y.iloc[train])
        probs = model.predict_proba(x.iloc[test])
        total.loc[probs.index] += probs
        counts.loc[probs.index] += 1
    return total.div(counts, axis=0)

In [ ]:
from __future__ import annotations

import hashlib
import json
from pathlib import Path

import joblib
import numpy as np
import pandas as pd

from IPython.display import display
from scipy.special import softmax
from scipy.spatial.distance import cdist

from sklearn.metrics import (
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    log_loss,
    roc_auc_score,
)
from sklearn.model_selection import RepeatedStratifiedKFold


CACHE_DIR = OUTPUT_DIR / "classifier_cache"
CACHE_DIR.mkdir(parents=True, exist_ok=True)

def atomic_joblib_dump(obj, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)

    temporary = path.with_name(path.name + ".tmp")
    joblib.dump(obj, temporary, compress=3)
    temporary.replace(path)


def atomic_to_csv(df, path, index=False):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)

    temporary = path.with_name(path.name + ".tmp")
    df.to_csv(temporary, index=index)
    temporary.replace(path)


def data_signature(x, y):
    """
    Create a cache identifier for the training dataset and settings.

    This prevents accidentally loading a cache generated from different
    samples, genes, or CV settings.
    """
    digest = hashlib.sha1()

    digest.update(str(x.shape).encode())
    digest.update("\n".join(x.index.astype(str)).encode())
    digest.update("\n".join(x.columns.astype(str)).encode())
    digest.update("\n".join(y.astype(str)).encode())

    configuration = {
        "class_order": list(CLASS_ORDER),
        "k_candidates": list(K_CANDIDATES),
        "cv_repeats": CV_REPEATS,
        "pca_variance": PCA_VARIANCE,
        "max_variable_features": MAX_VARIABLE_FEATURES,
        "random_state": RANDOM_STATE,
    }

    digest.update(
        json.dumps(configuration, sort_keys=True).encode()
    )

    return digest.hexdigest()[:16]

def tumor_training_data(cancer):
    meta = tumor_metadata[
        tumor_metadata["cancer"] == cancer
    ].copy()

    ids = meta.loc[
        meta["aligned_id"].isin(aligned_expression.index),
        "aligned_id",
    ]

    x = aligned_expression.loc[ids].copy()

    y = (
        meta.set_index("aligned_id")
        .loc[ids, "group"]
        .astype(str)
    )
    y.index = x.index

    print(
        f"{cancer}: {x.shape[0]} tumors × "
        f"{x.shape[1]} expression features"
    )
    print(y.value_counts())

    return x, y


def selected_cell_lines(codes):
    selected = (
        models[models["OncotreeCode"].isin(codes)]
        .drop_duplicates("ModelID", keep="first")
        .copy()
    )

    selected["aligned_id"] = (
        "CL::" + selected["ModelID"].astype(str)
    )

    selected = selected[
        selected["aligned_id"].isin(aligned_expression.index)
    ].copy()

    x = aligned_expression.loc[selected["aligned_id"]].copy()
    x.index = selected["aligned_id"].to_numpy()

    return x, selected

def make_cv(y):
    minimum_class_size = int(y.value_counts().min())
    n_splits = min(5, minimum_class_size)

    if n_splits < 2:
        raise ValueError(
            "At least two tumors per class are required."
        )

    return RepeatedStratifiedKFold(
        n_splits=n_splits,
        n_repeats=CV_REPEATS,
        random_state=RANDOM_STATE,
    )


def distances_for_all_k(
    fitted_model,
    x_test,
    k_values,
):
    """
    Transform the test samples once, calculate pairwise distances once,
    and derive class distances for every k.
    """
    test_coordinates = fitted_model._transform(x_test)

    pairwise_distances = cdist(
        test_coordinates,
        fitted_model.training_coordinates_,
        metric="cosine",
    )

    pairwise_distances = np.nan_to_num(
        pairwise_distances,
        nan=1.0,
        posinf=1.0,
        neginf=0.0,
    )

    results = {}

    for k in k_values:
        class_distances = []

        for class_name in CLASS_ORDER:
            class_mask = (
                fitted_model.training_labels_ == class_name
            )

            distances = pairwise_distances[:, class_mask]

            if distances.shape[1] == 0:
                raise ValueError(
                    f"No training samples for {class_name}"
                )

            k_used = min(int(k), distances.shape[1])

            nearest = np.partition(
                distances,
                kth=k_used - 1,
                axis=1,
            )[:, :k_used]

            class_distances.append(nearest.mean(axis=1))

        results[int(k)] = np.column_stack(class_distances)

    return results


def build_cv_distance_cache(
    cancer,
    x,
    y,
    force=False,
):
    """
    Build or recover one cache file per cross-validation fold.

    After an interruption, completed folds are loaded and only unfinished
    folds are calculated.
    """
    signature = data_signature(x, y)

    fold_dir = (
        CACHE_DIR
        / cancer.lower()
        / signature
        / "cv_folds"
    )
    fold_dir.mkdir(parents=True, exist_ok=True)

    cv = make_cv(y)
    splits = list(cv.split(x, y))

    fold_results = []

    print(
        f"{cancer}: {len(splits)} CV folds; "
        f"cache directory: {fold_dir}"
    )

    for fold_number, (train_idx, test_idx) in enumerate(splits):
        fold_path = fold_dir / f"fold_{fold_number:03d}.joblib"

        expected_test_ids = x.index[test_idx].astype(str).tolist()

        if fold_path.exists() and not force:
            try:
                payload = joblib.load(fold_path)

                valid = (
                    payload.get("signature") == signature
                    and payload.get("test_ids")
                    == expected_test_ids
                )

                if valid:
                    fold_results.append(payload)
                    print(
                        f"Fold {fold_number + 1}/{len(splits)}: "
                        "loaded"
                    )
                    continue

                print(
                    f"Fold {fold_number + 1}: "
                    "cache did not match; recalculating"
                )

            except Exception as error:
                print(
                    f"Fold {fold_number + 1}: "
                    f"could not load cache ({error}); recalculating"
                )

        print(
            f"Fold {fold_number + 1}/{len(splits)}: fitting"
        )

        model = BalancedNearestGroupClassifier(
            k=1,
            temperature=1.0,
            pca_variance=PCA_VARIANCE,
            max_features=MAX_VARIABLE_FEATURES,
        )

        model.fit(
            x.iloc[train_idx],
            y.iloc[train_idx],
        )

        distances_by_k = distances_for_all_k(
            fitted_model=model,
            x_test=x.iloc[test_idx],
            k_values=K_CANDIDATES,
        )

        payload = {
            "signature": signature,
            "fold_number": fold_number,
            "test_ids": expected_test_ids,
            "y_true": y.iloc[test_idx].astype(str).tolist(),
            "distances_by_k": distances_by_k,
        }

        
        atomic_joblib_dump(payload, fold_path)
        fold_results.append(payload)

    return fold_results, signature

def tune_from_cached_distances(
    cancer,
    folds,
    signature,
    force=False,
):
    result_path = (
        CACHE_DIR
        / cancer.lower()
        / signature
        / "tuning.csv"
    )

    if result_path.exists() and not force:
        print(f"Loading cached tuning: {result_path}")
        return pd.read_csv(result_path)

    rows = []

    for k in K_CANDIDATES:
        for temperature in TEMPERATURE_CANDIDATES:
            balanced_accuracy_values = []
            macro_f1_values = []
            log_loss_values = []

            for fold in folds:
                distances = fold["distances_by_k"][int(k)]
                y_true = np.asarray(fold["y_true"])

                probabilities = softmax(
                    -distances / float(temperature),
                    axis=1,
                )

                predictions = np.asarray(CLASS_ORDER)[
                    probabilities.argmax(axis=1)
                ]

                balanced_accuracy_values.append(
                    balanced_accuracy_score(
                        y_true,
                        predictions,
                    )
                )

                macro_f1_values.append(
                    f1_score(
                        y_true,
                        predictions,
                        labels=CLASS_ORDER,
                        average="macro",
                        zero_division=0,
                    )
                )

                log_loss_values.append(
                    log_loss(
                        y_true,
                        probabilities,
                        labels=CLASS_ORDER,
                    )
                )

            rows.append({
                "k": int(k),
                "temperature": float(temperature),
                "balanced_accuracy": np.mean(
                    balanced_accuracy_values
                ),
                "macro_f1": np.mean(macro_f1_values),
                "log_loss": np.mean(log_loss_values),
            })

    tuning = (
        pd.DataFrame(rows)
        .sort_values(
            [
                "balanced_accuracy",
                "macro_f1",
                "log_loss",
            ],
            ascending=[False, False, True],
        )
        .reset_index(drop=True)
    )

    atomic_to_csv(tuning, result_path, index=False)

    return tuning

def oof_from_cached_distances(
    x,
    folds,
    k,
    temperature,
):
    probability_sum = pd.DataFrame(
        0.0,
        index=x.index,
        columns=CLASS_ORDER,
    )

    prediction_count = pd.Series(
        0,
        index=x.index,
        dtype=int,
    )

    for fold in folds:
        test_ids = fold["test_ids"]
        distances = fold["distances_by_k"][int(k)]

        probabilities = softmax(
            -distances / float(temperature),
            axis=1,
        )

        probability_sum.loc[
            test_ids,
            CLASS_ORDER,
        ] += probabilities

        prediction_count.loc[test_ids] += 1

    if (prediction_count == 0).any():
        missing = prediction_count[
            prediction_count == 0
        ].index.tolist()

        raise RuntimeError(
            f"Samples without OOF predictions: {missing}"
        )

    return probability_sum.div(
        prediction_count,
        axis=0,
    )

def train_and_predict_cached(
    cancer,
    codes,
    fixed_parameters=None,
    calculate_oof=True,
    known_metrics=None,
    force=False,
):
    """
    Parameters
    ----------
    fixed_parameters
        Optional tuple: (k, temperature). When supplied, hyperparameter
        tuning is skipped.

    calculate_oof
        If False, skip repeated CV and only fit the final model.

    known_metrics
        Optional dictionary containing previously calculated metrics,
        useful when calculate_oof=False.
    """
    x_tumor, y_tumor = tumor_training_data(cancer)
    signature = data_signature(x_tumor, y_tumor)

    analysis_dir = (
        CACHE_DIR
        / cancer.lower()
        / signature
    )
    analysis_dir.mkdir(parents=True, exist_ok=True)

    folds = None
    tuning = None
    oof = None

    
    if fixed_parameters is None or calculate_oof:
        folds, signature = build_cv_distance_cache(
            cancer=cancer,
            x=x_tumor,
            y=y_tumor,
            force=force,
        )

    if fixed_parameters is None:
        tuning = tune_from_cached_distances(
            cancer=cancer,
            folds=folds,
            signature=signature,
            force=force,
        )

        best = tuning.iloc[0]
        k = int(best["k"])
        temperature = float(best["temperature"])

    else:
        k = int(fixed_parameters[0])
        temperature = float(fixed_parameters[1])

        tuning = pd.DataFrame([{
            "k": k,
            "temperature": temperature,
            "fixed_parameters": True,
        }])

    print(
        f"{cancer}: selected k={k}, "
        f"temperature={temperature}"
    )

    performance_path = analysis_dir / "performance.csv"
    oof_path = analysis_dir / (
        f"oof_k{k}_temperature{temperature}.pkl"
    )

    if calculate_oof:
        if oof_path.exists() and not force:
            print(f"Loading cached OOF: {oof_path}")
            oof = pd.read_pickle(oof_path)
        else:
            oof = oof_from_cached_distances(
                x=x_tumor,
                folds=folds,
                k=k,
                temperature=temperature,
            )

            temporary = oof_path.with_name(
                oof_path.name + ".tmp"
            )
            oof.to_pickle(temporary)
            temporary.replace(oof_path)

        oof_pred = oof.idxmax(axis=1)

        performance_values = {
            "cancer": cancer,
            "k": k,
            "temperature": temperature,
            "balanced_accuracy": balanced_accuracy_score(
                y_tumor,
                oof_pred,
            ),
            "macro_f1": f1_score(
                y_tumor,
                oof_pred,
                labels=CLASS_ORDER,
                average="macro",
                zero_division=0,
            ),
        }

        auc_order = sorted(CLASS_ORDER)

        try:
            performance_values["macro_ovr_auc"] = (
                roc_auc_score(
                    y_tumor,
                    oof[auc_order],
                    labels=auc_order,
                    multi_class="ovr",
                    average="macro",
                )
            )
        except ValueError:
            performance_values["macro_ovr_auc"] = np.nan

        performance = pd.DataFrame(
            [performance_values]
        )

        print(f"\n{cancer} repeated cross-validation")
        display(performance)

        display(
            pd.DataFrame(
                confusion_matrix(
                    y_tumor,
                    oof_pred,
                    labels=CLASS_ORDER,
                ),
                index=[
                    f"true_{value}"
                    for value in CLASS_ORDER
                ],
                columns=[
                    f"pred_{value}"
                    for value in CLASS_ORDER
                ],
            )
        )

        print(
            classification_report(
                y_tumor,
                oof_pred,
                labels=CLASS_ORDER,
                zero_division=0,
            )
        )

        oof_export = oof.copy()
        oof_export["true_group"] = y_tumor
        oof_export["predicted_group"] = oof_pred
        oof_export["cancer"] = cancer
        oof_export.index.name = "aligned_id"
        oof_export = oof_export.reset_index()

        atomic_to_csv(
            oof_export,
            analysis_dir / "oof_predictions.csv",
            index=False,
        )

    else:
        performance_values = {
            "cancer": cancer,
            "k": k,
            "temperature": temperature,
            "balanced_accuracy": np.nan,
            "macro_f1": np.nan,
            "macro_ovr_auc": np.nan,
        }

        if known_metrics is not None:
            performance_values.update(known_metrics)

        performance = pd.DataFrame(
            [performance_values]
        )
        oof_export = pd.DataFrame()

    atomic_to_csv(
        performance,
        performance_path,
        index=False,
    )

    
    
    

    model_path = analysis_dir / (
        f"final_model_k{k}_temperature{temperature}.joblib"
    )

    if model_path.exists() and not force:
        try:
            print(f"Loading final model: {model_path}")
            final_model = joblib.load(model_path)
        except Exception as error:
            print(
                f"Could not load final model ({error}); refitting"
            )
            final_model = None
    else:
        final_model = None

    if final_model is None:
        print(f"{cancer}: fitting final model")

        final_model = BalancedNearestGroupClassifier(
            k=k,
            temperature=temperature,
            pca_variance=PCA_VARIANCE,
            max_features=MAX_VARIABLE_FEATURES,
        ).fit(
            x_tumor,
            y_tumor,
        )

        atomic_joblib_dump(
            final_model,
            model_path,
        )

    
    
    

    code_hash = hashlib.sha1(
        ",".join(sorted(codes)).encode()
    ).hexdigest()[:8]

    predictions_path = analysis_dir / (
        f"cell_line_predictions_{code_hash}.csv"
    )

    if predictions_path.exists() and not force:
        print(
            f"Loading cell-line predictions: "
            f"{predictions_path}"
        )
        predictions = pd.read_csv(predictions_path)

    else:
        x_cell, selected = selected_cell_lines(codes)

        probabilities = final_model.predict_proba(x_cell)

        predictions = probabilities.rename(
            columns={
                "MSS/TMB-L": "p_MSS_TMBL",
                "MSS/TMB-H": "p_MSS_TMBH",
                "MSI-H": "p_MSIH",
            }
        )

        predictions["predicted_group"] = (
            probabilities.idxmax(axis=1)
        )

        sorted_probabilities = np.sort(
            probabilities.to_numpy(),
            axis=1,
        )

        predictions["confidence"] = (
            sorted_probabilities[:, -1]
        )

        predictions["probability_margin"] = (
            sorted_probabilities[:, -1]
            - sorted_probabilities[:, -2]
        )

        predictions["DepMap_ID"] = (
            predictions.index
            .str.replace(
                r"^CL::",
                "",
                regex=True,
            )
        )

        
        selected_metadata = (
            selected
            .drop(columns="aligned_id")
            .rename(columns={"ModelID": "DepMap_ID"})
        )

        predictions = predictions.merge(
            selected_metadata,
            on="DepMap_ID",
            how="left",
            validate="one_to_one",
        )

        predictions.insert(
            0,
            "classifier_source",
            cancer,
        )

        atomic_to_csv(
            predictions,
            predictions_path,
            index=False,
        )

    return {
        "model": final_model,
        "tuning": tuning,
        "performance": performance,
        "oof": oof_export,
        "predictions": predictions,
        "cache_directory": analysis_dir,
    }

In [ ]:
crc_analysis = train_and_predict_cached(
    cancer="CRC",
    codes=CRC_ONCOTREE_CODES,
    fixed_parameters=(1, 0.05),
    calculate_oof=False,
    known_metrics={
        "balanced_accuracy": 0.614235,
        "macro_f1": 0.623277,
        "macro_ovr_auc": 0.913421,
    },
)

crc_predictions = crc_analysis["predictions"].copy()

In [ ]:
ucec_analysis = train_and_predict_cached(
    cancer="UCEC",
    codes=UTERINE_BROAD_CODES,
    fixed_parameters=None,
    calculate_oof=True,
)

uterine_broad_predictions = (
    ucec_analysis["predictions"].copy()
)

uterine_carcinoma_predictions = (
    uterine_broad_predictions[
        uterine_broad_predictions["OncotreeCode"].isin(
            UTERINE_CARCINOMA_CODES
        )
    ].copy()
)

In [ ]:
classifier_performance = pd.concat(
    [
        crc_analysis["performance"],
        ucec_analysis["performance"],
    ],
    ignore_index=True,
)

display(classifier_performance)

In [ ]:
prediction_sets = {
    "CRC": crc_predictions,
    "Uterine carcinoma": uterine_carcinoma_predictions,
    "Uterine broad": uterine_broad_predictions,
}

for cohort, pred in prediction_sets.items():
    pred["high_confidence"] = (
        (pred["confidence"] >= PRIMARY_PROBABILITY_THRESHOLD)
        & (pred["probability_margin"] >= PRIMARY_MARGIN_THRESHOLD)
    )
    print(f"\n{cohort}")
    display(pred.groupby(["predicted_group", "high_confidence"]).size().rename("n").reset_index())

pd.concat(
    [x.assign(analysis_cohort=k) for k, x in prediction_sets.items()],
    ignore_index=True,
).to_csv(OUTPUT_DIR / "cell_line_subgroup_predictions.csv", index=False)

In [ ]:
def first_non_null(series):
    values = series.dropna()
    return values.iloc[0] if len(values) else np.nan


def prepare_prism(prism, treatment_info=None, response_col="auc"):
    required = {"depmap_id", "name", response_col}
    missing = required.difference(prism.columns)
    if missing:
        raise KeyError(f"Missing PRISM columns: {sorted(missing)}")

    data = prism.copy()
    data["drug_name"] = data["name"].astype(str).str.strip()
    data["drug_id"] = data["drug_name"].str.lower()
    data[response_col] = pd.to_numeric(data[response_col], errors="coerce")
    data = data.dropna(subset=["depmap_id", "drug_id", response_col])

    collapsed = data.groupby(["depmap_id", "drug_id"], as_index=False).agg(
        drug_name=("drug_name", "first"),
        auc=(response_col, "median"),
        n_measurements=(response_col, "size"),
    )

    columns = ["name", "broad_id", "moa", "target", "phase", "disease.area", "indication"]
    parts = []
    for source in [prism, treatment_info]:
        if source is None or "name" not in source.columns:
            continue
        available = [c for c in columns if c in source.columns]
        meta = source[available].copy()
        meta["drug_id"] = meta["name"].astype(str).str.strip().str.lower()
        parts.append(meta)

    if parts:
        meta = pd.concat(parts, ignore_index=True)
        meta = meta.groupby("drug_id", as_index=False).agg({
            c: first_non_null for c in meta.columns if c != "drug_id"
        }).rename(columns={"name": "drug_name_metadata"})
    else:
        meta = collapsed[["drug_id", "drug_name"]].drop_duplicates()

    return collapsed, meta


prism_collapsed, drug_metadata = prepare_prism(prism, treatment_info)
print("Unique compounds:", prism_collapsed["drug_id"].nunique())
print("Unique cell lines:", prism_collapsed["depmap_id"].nunique())

In [ ]:
classified = crc_predictions[
        (crc_predictions["confidence"] >= PRIMARY_PROBABILITY_THRESHOLD)
        & (crc_predictions["probability_margin"] >= PRIMARY_MARGIN_THRESHOLD)
    ][["DepMap_ID", "predicted_group"]].drop_duplicates("DepMap_ID")
merged = classified.merge(
        prism_collapsed, left_on="DepMap_ID", right_on="depmap_id", how="inner"
    )
rows = []
for drug_id, drug_df in merged.groupby("drug_id"):
    mss = drug_df.loc[drug_df["predicted_group"] == "MSS/TMB-H", "auc"].dropna()

    for comparator in ["MSS/TMB-L", "MSI-H"]:
        ref = drug_df.loc[drug_df["predicted_group"] == comparator, "auc"].dropna()
        if len(mss) < MIN_MODELS_PER_GROUP or len(ref) < MIN_MODELS_PER_GROUP:
            continue
        u, p = mannwhitneyu(mss, ref, alternative="two-sided", method="auto")
        rows.append({
            "cohort": cohort,
            "probability_threshold": PRIMARY_PROBABILITY_THRESHOLD,
            "drug_id": drug_id,
            "drug": drug_df["drug_name"].iloc[0],
            "contrast": f"MSS/TMB-H vs {comparator}",
            "n_MSS_TMBH": len(mss),
            "n_comparator": len(ref),
            "median_auc_MSS_TMBH": np.median(mss),
            "median_auc_comparator": np.median(ref),
            "median_auc_difference": np.median(mss) - np.median(ref),
            "pvalue": p,
        })
pd.DataFrame(rows)

In [ ]:
print("All predicted cell lines:")
display(
    crc_predictions["predicted_group"]
    .value_counts(dropna=False)
    .rename("n")
)

classified = crc_predictions[
    (crc_predictions["confidence"] >= PRIMARY_PROBABILITY_THRESHOLD)
    & (
        crc_predictions["probability_margin"]
        >= PRIMARY_MARGIN_THRESHOLD
    )
].copy()

print("\nAfter confidence thresholds:")
display(
    classified["predicted_group"]
    .value_counts(dropna=False)
    .rename("n")
)

merged = classified.merge(
    prism_collapsed,
    left_on="DepMap_ID",
    right_on="depmap_id",
    how="inner",
)

print("\nAfter merging with PRISM:")
display(
    merged["predicted_group"]
    .value_counts(dropna=False)
    .rename("n")
)

print("\nUnique models with PRISM data:")
display(
    merged.groupby("predicted_group")["DepMap_ID"]
    .nunique()
    .rename("n_models")
)

In [ ]:
mss

In [ ]:
def cliffs_delta_from_u(u, nx, ny):
    return (2.0 * u / (nx * ny)) - 1.0


def screen_hard_groups(predictions, cohort, threshold):
    classified = predictions[
        (predictions["confidence"] >= threshold)
        & (predictions["probability_margin"] >= PRIMARY_MARGIN_THRESHOLD)
    ][["DepMap_ID", "predicted_group"]].drop_duplicates("DepMap_ID")

    merged = classified.merge(
        prism_collapsed, left_on="DepMap_ID", right_on="depmap_id", how="inner"
    )
    rows = []
    for drug_id, drug_df in merged.groupby("drug_id"):
        mss = drug_df.loc[drug_df["predicted_group"] == "MSS/TMB-H", "auc"].dropna()
        for comparator in ["MSS/TMB-L", "MSI-H"]:
            ref = drug_df.loc[drug_df["predicted_group"] == comparator, "auc"].dropna()
            if len(mss) < MIN_MODELS_PER_GROUP or len(ref) < MIN_MODELS_PER_GROUP:
                continue
            u, p = mannwhitneyu(mss, ref, alternative="two-sided", method="auto")
            rows.append({
                "cohort": cohort,
                "probability_threshold": threshold,
                "drug_id": drug_id,
                "drug": drug_df["drug_name"].iloc[0],
                "contrast": f"MSS/TMB-H vs {comparator}",
                "n_MSS_TMBH": len(mss),
                "n_comparator": len(ref),
                "median_auc_MSS_TMBH": np.median(mss),
                "median_auc_comparator": np.median(ref),
                "median_auc_difference": np.median(mss) - np.median(ref),
                "cliffs_delta": cliffs_delta_from_u(u, len(mss), len(ref)),
                "pvalue": p,
            })
    return pd.DataFrame(rows)


primary_hard_results = pd.concat([
    screen_hard_groups(crc_predictions, "CRC", PRIMARY_PROBABILITY_THRESHOLD),
    screen_hard_groups(uterine_carcinoma_predictions, "Uterine carcinoma",
                       PRIMARY_PROBABILITY_THRESHOLD),
    screen_hard_groups(uterine_broad_predictions, "Uterine broad",
                       PRIMARY_PROBABILITY_THRESHOLD),
], ignore_index=True)

if not primary_hard_results.empty:
    primary_hard_results["FDR_global"] = multipletests(
        primary_hard_results["pvalue"], method="fdr_bh"
    )[1]
    primary_hard_results["FDR_within_cohort_contrast"] = (
        primary_hard_results.groupby(["cohort", "contrast"])["pvalue"]
        .transform(lambda p: multipletests(p, method="fdr_bh")[1])
    )
    primary_hard_results = primary_hard_results.merge(
        drug_metadata, on="drug_id", how="left"
    ).sort_values(["FDR_within_cohort_contrast", "pvalue"])

display(primary_hard_results.head(30))
primary_hard_results.to_csv(OUTPUT_DIR / "all_drug_hard_group_results.csv", index=False)

In [ ]:
def screen_probability(predictions, cohort):
    scores = predictions[["DepMap_ID", "p_MSS_TMBH"]].drop_duplicates("DepMap_ID")
    merged = scores.merge(
        prism_collapsed, left_on="DepMap_ID", right_on="depmap_id", how="inner"
    )
    rows = []
    for drug_id, drug_df in merged.groupby("drug_id"):
        test = drug_df[["p_MSS_TMBH", "auc"]].dropna()
        if len(test) < MIN_MODELS_CONTINUOUS or test["p_MSS_TMBH"].nunique() < 3:
            continue
        rho, p = spearmanr(test["p_MSS_TMBH"], test["auc"])
        rows.append({
            "cohort": cohort, "drug_id": drug_id,
            "drug": drug_df["drug_name"].iloc[0],
            "n": len(test), "rho_probability_auc": rho, "pvalue": p,
        })
    return pd.DataFrame(rows)


continuous_results = pd.concat([
    screen_probability(crc_predictions, "CRC"),
    screen_probability(uterine_carcinoma_predictions, "Uterine carcinoma"),
    screen_probability(uterine_broad_predictions, "Uterine broad"),
], ignore_index=True)

if not continuous_results.empty:
    continuous_results["FDR_global"] = multipletests(
        continuous_results["pvalue"], method="fdr_bh"
    )[1]
    continuous_results["FDR_within_cohort"] = (
        continuous_results.groupby("cohort")["pvalue"]
        .transform(lambda p: multipletests(p, method="fdr_bh")[1])
    )
    continuous_results = continuous_results.merge(
        drug_metadata, on="drug_id", how="left"
    ).sort_values(["FDR_within_cohort", "pvalue"])

display(continuous_results.head(30))
continuous_results.to_csv(
    OUTPUT_DIR / "all_drug_continuous_probability_results.csv", index=False
)

In [ ]:
def prioritize_specific(results):
    if results.empty:
        return pd.DataFrame()

    significant = results[
        (results["FDR_within_cohort_contrast"] < FDR_CUTOFF)
        & (results["median_auc_difference"] < 0)
    ].copy()

    a = significant[significant["contrast"] == "MSS/TMB-H vs MSS/TMB-L"]
    b = significant[significant["contrast"] == "MSS/TMB-H vs MSI-H"]
    cols = [
        "cohort", "drug_id", "drug", "median_auc_difference",
        "cliffs_delta", "pvalue", "FDR_within_cohort_contrast",
    ]
    candidates = a[cols].merge(
        b[cols], on=["cohort", "drug_id", "drug"],
        suffixes=("_vs_MSS_TMBL", "_vs_MSIH"), how="inner"
    )
    if candidates.empty:
        return candidates

    candidates["worst_FDR"] = candidates[[
        "FDR_within_cohort_contrast_vs_MSS_TMBL",
        "FDR_within_cohort_contrast_vs_MSIH",
    ]].max(axis=1)
    candidates["conservative_auc_advantage"] = -candidates[[
        "median_auc_difference_vs_MSS_TMBL",
        "median_auc_difference_vs_MSIH",
    ]].max(axis=1)

    metadata_cols = [c for c in [
        "drug_name_metadata", "broad_id", "moa", "target", "phase",
        "disease.area", "indication"
    ] if c in drug_metadata.columns]
    candidates = candidates.merge(
        drug_metadata[["drug_id"] + metadata_cols], on="drug_id", how="left"
    )
    return candidates.sort_values(
        ["worst_FDR", "conservative_auc_advantage"], ascending=[True, False]
    )


specific_candidates = prioritize_specific(primary_hard_results)
display(specific_candidates)
specific_candidates.to_csv(
    OUTPUT_DIR / "mss_tmbh_specific_candidate_drugs.csv", index=False
)

In [ ]:
threshold_parts = []
for threshold in THRESHOLDS_TO_TEST:
    threshold_parts.extend([
        screen_hard_groups(crc_predictions, "CRC", threshold),
        screen_hard_groups(uterine_carcinoma_predictions, "Uterine carcinoma", threshold),
        screen_hard_groups(uterine_broad_predictions, "Uterine broad", threshold),
    ])

threshold_results = pd.concat(threshold_parts, ignore_index=True)
if not threshold_results.empty:
    threshold_results["FDR_within_threshold_cohort_contrast"] = (
        threshold_results.groupby(
            ["probability_threshold", "cohort", "contrast"]
        )["pvalue"].transform(lambda p: multipletests(p, method="fdr_bh")[1])
    )
    threshold_results["significant_sensitive"] = (
        (threshold_results["FDR_within_threshold_cohort_contrast"] < FDR_CUTOFF)
        & (threshold_results["median_auc_difference"] < 0)
    )
    threshold_summary = threshold_results.groupby(
        ["cohort", "drug_id", "drug", "contrast"]
    ).agg(
        thresholds_tested=("probability_threshold", "nunique"),
        thresholds_significant=("significant_sensitive", "sum"),
        all_effects_sensitive=("median_auc_difference", lambda x: bool((x < 0).all())),
        median_effect=("median_auc_difference", "median"),
    ).reset_index().sort_values(
        ["thresholds_significant", "median_effect"], ascending=[False, True]
    )
else:
    threshold_summary = pd.DataFrame()

display(threshold_summary.head(50))
threshold_results.to_csv(
    OUTPUT_DIR / "all_drug_threshold_sensitivity_results.csv", index=False
)
threshold_summary.to_csv(
    OUTPUT_DIR / "all_drug_threshold_sensitivity_summary.csv", index=False
)

In [ ]:
primary_hard_results

In [ ]:
def cross_cancer_table(results, contrast):
    subset = results[
        (results["contrast"] == contrast)
        & results["cohort"].isin(["CRC", "Uterine carcinoma"])
    ]
    if subset.empty:
        return pd.DataFrame()
    pivot = subset.pivot_table(
        index=["drug_id", "drug"], columns="cohort",
        values=["median_auc_difference", "FDR_within_cohort_contrast"],
        aggfunc="first",
    )
    pivot.columns = [
        f"{metric}_{cohort.replace(' ', '_')}" for metric, cohort in pivot.columns
    ]
    pivot = pivot.reset_index()
    a = "median_auc_difference_CRC"
    b = "median_auc_difference_Uterine_carcinoma"
    if a in pivot.columns and b in pivot.columns:
        pivot["same_direction"] = np.sign(pivot[a]) == np.sign(pivot[b])
        pivot["both_more_sensitive"] = (pivot[a] < 0) & (pivot[b] < 0)
    return pivot.sort_values(
        ["both_more_sensitive", "same_direction"], ascending=[False, False]
    )


cross_mss = cross_cancer_table(primary_hard_results, "MSS/TMB-H vs MSS/TMB-L")
cross_msi = cross_cancer_table(primary_hard_results, "MSS/TMB-H vs MSI-H")
display(cross_mss.head(30))
display(cross_msi.head(30))
cross_mss.to_csv(OUTPUT_DIR / "cross_cancer_vs_mss_tmbl.csv", index=False)
cross_msi.to_csv(OUTPUT_DIR / "cross_cancer_vs_msih.csv", index=False)

In [ ]:
classifier_performance.to_csv(OUTPUT_DIR / "classifier_performance.csv", index=False)
crc_analysis["tuning"].to_csv(OUTPUT_DIR / "crc_classifier_tuning.csv", index=False)
ucec_analysis["tuning"].to_csv(OUTPUT_DIR / "ucec_classifier_tuning.csv", index=False)
crc_analysis["oof"].to_csv(OUTPUT_DIR / "crc_tumor_oof_predictions.csv", index=False)
ucec_analysis["oof"].to_csv(OUTPUT_DIR / "ucec_tumor_oof_predictions.csv", index=False)

print("Outputs written to:", OUTPUT_DIR)
for path in sorted(OUTPUT_DIR.glob("*")):
    print(" -", path.name)